# Word2Vec — Assignment 3: using unlabelled data with word embeddings

In this assignment, we will see how we can use Word2Vec (or any similar word embedding) to use information from unlabelled data to help us classify better!

You will be using the sentiment data from last week, either the yelps or movies, whichever you wish.

Your goal will be to simulate the following situation: you have a **small** set of labelled data and a large set of unlabelled data. Show how the two follow 2 techniques compare as the amount of labelled data increases. You should train them on the small labelled subset and test their performance on the rest of the data.

In other words, train on 1k, test on 99k. Then train on 2k, test on 98k. Then train on 4k, test on 96k. Etc.

1. Logistic regression trained on labelled data, documents represented as term-frequency matrix of your choice. You can learn the vocabulary from the entire dataset or only the labelled data.

2. Logistic regression trained on the labelled data, documents represented as word2vec vectors where you train word2vec using the entire dataset. Play around with different settings of word2vec (training window size, K-negative, skip-gram vs BOW, training windows, etc.). Note: we didn't go over the options in detail in class, so you will need to read about them a bit!

You can read about the gensim word2vec implementation [here](https://radimrehurek.com/gensim/models/word2vec.html).

## Overview

**Data:** Yelp (100k) and IMDB (50k) reviews combined into one sentiment dataset of about 150k documents. Random subsets of 1%, 2% and 4% are treated as *labelled*; the rest is the test set.

**Approaches:**

1. **TF-IDF + logistic regression:** `TfidfVectorizer` and `LogisticRegression` in a `Pipeline`, tuned with `GridSearchCV` (5-fold CV) on each labelled subset.
2. **Word2Vec + logistic regression:** each review is represented as the average of its gensim `Word2Vec` word vectors. Reviews with no known words are filled in with `KNNImputer`.

**Tools:** pandas, NumPy, NLTK (`WordNetLemmatizer`), scikit-learn, gensim.

| Labelled data | TF-IDF + LR (test accuracy) | Word2Vec + LR (test accuracy) |
|---|---|---|
| 1% (~1.6k reviews) | 0.872 | 0.67 |
| 2% | 0.887 | – |
| 4% | 0.902 | – |

TF-IDF improves steadily as more labelled data is added. Word2Vec scored much lower in the saved run because it was trained on the 1% labelled subset only (see the note in the Word2Vec section). The **Extension** at the end trains Word2Vec on all reviews, as the assignment intends.

## Load, preprocess and split the data

In [ ]:
import os

# On Google Colab, mount Drive and move to the folder with the data.
# Locally, keep yelps.csv and movies.csv next to this notebook.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/My Drive/Colab Notebooks')  # change to your own folder
except ImportError:
    pass

Mounted at /content/drive


In [ ]:
import re

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

yelps = pd.read_csv('yelps.csv').fillna(' ')
imdb = pd.read_csv('movies.csv').fillna(' ')

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4', quiet=True)  # newer NLTK versions also need this for WordNet

lemmatizer = WordNetLemmatizer()


def preprocess(texts):
    """Regex cleaning + WordNet lemmatisation (spaCy lemmatisation was too slow for this many reviews)."""
    documents = []
    for text in texts:
        document = re.sub(r'\W', ' ', str(text))             # remove special characters
        document = re.sub(r'\s+[a-zA-Z]\s+', ' ', document)  # remove single characters
        document = re.sub(r'\^[a-zA-Z]\s+', ' ', document)   # meant for a leading single character (as written, matches a literal '^')
        document = re.sub(r'\s+', ' ', document)             # collapse whitespace
        document = re.sub(r'^b\s+', '', document)            # remove the 'b' prefix of byte strings
        document = document.lower()
        document = ' '.join(lemmatizer.lemmatize(word) for word in document.split())
        document = re.sub(r'\b\d+(?:\.\d+)?\s+', '', document)  # remove numbers
        documents.append(document)
    return documents


df = pd.concat([yelps[['positive', 'text']], imdb[['positive', 'text']]]).reset_index(drop=True)
df['clean'] = preprocess(df.text)  # preprocess once; every split reuses it

# Draw a random "labelled" subset of ~1%, 2% and 4%; everything else is the test set.
# The seed was added later for reproducibility; the saved outputs come from an unseeded run.
np.random.seed(42)
TRAIN_PCTS = [1, 2, 4]
splits = {}
for pct in TRAIN_PCTS:
    mask = np.random.rand(len(df)) < pct / 100
    splits[pct] = {
        'mask': mask,
        'X_train': list(df.clean[mask]), 'y_train': df.positive[mask],
        'X_test': list(df.clean[~mask]), 'y_test': df.positive[~mask],
    }

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Unzipping corpora/wordnet.zip.


In [ ]:
print(len(splits[1]['y_train']))
print(len(splits[1]['X_train']))

1611
1611


In [ ]:
print(len(yelps))
print(len(imdb))

100003
50003


> **Note on preprocessing:** an earlier version also had a dictionary for expanding contractions (*don't* → *do not*). It ran *after* punctuation had been stripped, so it never matched anything. It has been removed, which has no effect on the results. Expanding contractions *before* removing punctuation would be a reasonable improvement.

# Logistic regression on TF-IDF features

Different feature extractors and classifiers could be compared, but that is slow, so only the `TfidfVectorizer` and `LogisticRegression` hyperparameters are tuned here (5-fold CV on each labelled subset).

In [ ]:
tvc_pipe = Pipeline([
    ('tvec', TfidfVectorizer()),
    ('lr', LogisticRegression()),
])

tf_params = {
    'tvec__min_df': [2, 5, 10, 12, 15, 18, 20],
    'tvec__max_df': [0.8, 0.82, 0.85, 0.78, 0.75],
    'tvec__norm': ['l2'],
    'tvec__use_idf': [True],
    'lr__penalty': ['l2'],
}

tfidf_models = {}
for pct in TRAIN_PCTS:
    s = splits[pct]
    print(f'{pct}% train and {100 - pct}% test')
    gs = GridSearchCV(tvc_pipe, param_grid=tf_params, cv=5, verbose=1, n_jobs=-1)
    gs.fit(s['X_train'], s['y_train'])
    print(f"score for train data: {gs.score(s['X_train'], s['y_train'])}")
    print(f"score for test data: {gs.score(s['X_test'], s['y_test'])}")
    tfidf_models[pct] = gs

1% train and 99% test
Fitting 5 folds for each of 35 candidates, totalling 175 fits


[Parallel(n_jobs=100)]: Using backend LokyBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done 152 out of 175 | elapsed:  1.8min remaining:   16.3s
[Parallel(n_jobs=100)]: Done 175 out of 175 | elapsed:  1.8min finished


score for train data: 0.9484792054624457
score for test data: 0.8720711614272718
2% train and 98% test
Fitting 5 folds for each of 35 candidates, totalling 175 fits


[Parallel(n_jobs=100)]: Using backend LokyBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done 152 out of 175 | elapsed:  1.2min remaining:   11.2s
[Parallel(n_jobs=100)]: Done 175 out of 175 | elapsed:  1.3min finished


score for train data: 0.9528519617540389
score for test data: 0.8873398515373572
4% train and 96% test
Fitting 5 folds for each of 35 candidates, totalling 175 fits


[Parallel(n_jobs=100)]: Using backend LokyBackend with 100 concurrent workers.
[Parallel(n_jobs=100)]: Done 152 out of 175 | elapsed:  2.5min remaining:   22.5s
[Parallel(n_jobs=100)]: Done 175 out of 175 | elapsed:  2.7min finished


score for train data: 0.9436177248677249
score for test data: 0.9016240848025119


# Logistic regression on averaged Word2Vec vectors

> **Note:** in the saved run below, Word2Vec was trained on the **1% labelled subset only** (~1.6k reviews), with `min_count=40`. That gives a small vocabulary and noisy vectors, and it does not use the unlabelled reviews, which is the point of this assignment. The **Extension** section at the end trains Word2Vec on all reviews.

In [ ]:
train_tokens = [doc.split(' ') for doc in splits[1]['X_train']]

In [ ]:
from gensim.models import Word2Vec
from sklearn.impute import KNNImputer
from sklearn.metrics import classification_report

num_features = 100    # word vector dimensionality
min_word_count = 40   # ignore words that occur fewer times than this
num_workers = 3       # number of training threads
context = 10          # context window size
downsampling = 1e-3   # downsampling of very frequent words
model = Word2Vec(train_tokens, workers=num_workers, vector_size=num_features,
                 min_count=min_word_count, window=context, sample=downsampling)  # CBOW by default (sg=0)

In [ ]:
def average_word_vectors(token_lists, wv):
    """One row per document: the mean of its in-vocabulary word vectors (NaN if it has none)."""
    vecs = np.full((len(token_lists), wv.vector_size), np.nan, dtype='float32')
    for i, tokens in enumerate(token_lists):
        known = [t for t in tokens if t in wv.key_to_index]
        if known:
            vecs[i] = wv[known].mean(axis=0)
    return vecs

In [ ]:
train_vecs_1 = average_word_vectors(train_tokens, model.wv)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:12: DeprecationWarning: Call to deprecated `__getitem__` (Method will be removed in 4.0.0, use self.wv.__getitem__() instead).
  if sys.path[0] == '':


In [ ]:
test_tokens_99 = [doc.split(' ') for doc in splits[1]['X_test']]

In [ ]:
test_vecs_99 = average_word_vectors(test_tokens_99, model.wv)

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:12: DeprecationWarning: Call to deprecated `__getitem__` (Method will be removed in 4.0.0, use self.wv.__getitem__() instead).
  if sys.path[0] == '':
/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:14: RuntimeWarning: invalid value encountered in true_divide
  


In [ ]:
# Reviews with no word in the Word2Vec vocabulary have NaN vectors; fill them in from the 50 most similar reviews.
knn = KNNImputer(n_neighbors=50)
test_vecs_99 = knn.fit_transform(test_vecs_99)

In [ ]:
test_vecs_99.shape

(148395, 100)

In [ ]:
np.isnan(test_vecs_99).any()

False

In [ ]:
lr1 = LogisticRegression()
lr1.fit(train_vecs_1, splits[1]['y_train'])
result = lr1.predict(test_vecs_99)
print(classification_report(splits[1]['y_test'], result))

              precision    recall  f1-score   support

           0       0.66      0.68      0.67     74189
           1       0.67      0.65      0.66     74206

    accuracy                           0.67    148395
   macro avg       0.67      0.67      0.67    148395
weighted avg       0.67      0.67      0.67    148395



In [ ]:
print(classification_report(splits[1]['y_train'], lr1.predict(train_vecs_1)))

              precision    recall  f1-score   support

           0       0.68      0.70      0.69       817
           1       0.68      0.67      0.67       794

    accuracy                           0.68      1611
   macro avg       0.68      0.68      0.68      1611
weighted avg       0.68      0.68      0.68      1611



## Results

| Labelled data | TF-IDF + LR | Word2Vec (trained on the labelled 1% only) + LR |
|---|---|---|
| 1% | 0.872 | 0.67 |
| 2% | 0.887 | – |
| 4% | 0.902 | – |

* TF-IDF benefits steadily from more labelled data (+3 points from 1% to 4%).
* Averaged Word2Vec vectors trained on only ~1.6k reviews do much worse (0.67) and also fit the training data poorly (0.68), so the model is under-fitting. With so little text the embeddings are not informative.

## Extension: Word2Vec trained on all reviews (not run)

Word2Vec needs no labels, so it can learn from every review. The cells below train it once per configuration on the full corpus (about 150k reviews) and reuse it for every labelled split. They compare CBOW and skip-gram, two window sizes and two negative-sampling settings against the TF-IDF models above. *These cells were added after the course and have not been run, because the data is not in this repository.*

In [ ]:
all_tokens = [doc.split(' ') for doc in df.clean]
y_all = df.positive.values

w2v_configs = {
    'CBOW, window 5':       dict(sg=0, window=5, negative=5),
    'skip-gram, window 5':  dict(sg=1, window=5, negative=5),
    'skip-gram, window 10': dict(sg=1, window=10, negative=15),
}

results = [
    {'labelled %': pct, 'features': 'TF-IDF (grid search)',
     'test accuracy': tfidf_models[pct].score(s['X_test'], s['y_test'])}
    for pct, s in splits.items()
]

for name, cfg in w2v_configs.items():
    w2v = Word2Vec(all_tokens, vector_size=100, min_count=5, sample=1e-3, workers=4, seed=42, **cfg)
    doc_vecs = np.nan_to_num(average_word_vectors(all_tokens, w2v.wv))  # no known word -> zero vector
    for pct, s in splits.items():
        m = s['mask']
        clf = LogisticRegression(max_iter=1000).fit(doc_vecs[m], y_all[m])
        results.append({'labelled %': pct, 'features': f'Word2Vec ({name})',
                        'test accuracy': clf.score(doc_vecs[~m], y_all[~m])})

results = pd.DataFrame(results)
results.pivot(index='features', columns='labelled %', values='test accuracy').round(3)

In [ ]:
ax = results.pivot(index='labelled %', columns='features', values='test accuracy').plot(marker='o', figsize=(8, 5))
ax.set(xlabel='Labelled data (% of all reviews)', ylabel='Test accuracy',
       title='TF-IDF vs Word2Vec features as labelled data grows')
plt.show()